In [ ]:
import pickle
from pathlib import Path

import pandas as pd
import scipy.sparse as sparse
from sklearn.feature_extraction.text import TfidfVectorizer


# ============================================================================
# 03. FEATURE EXTRACTION
# ============================================================================

BASE_DIR = Path("..").resolve()
INPUT_CSV_PATH = BASE_DIR / "data/processed/combined_nli_masked.csv"
FEATURES_DIR = BASE_DIR / "data/processed/features"
MATRICES_PATH = BASE_DIR / "data/processed/nli_feature_matrices.pkl"

FEATURES_DIR.mkdir(parents=True, exist_ok=True)

print("Loading POS-masked corpus...")
df = pd.read_csv(INPUT_CSV_PATH)
df["masked_text"] = df["masked_text"].fillna("").astype(str)
df["dataset"] = df["dataset"].astype(str).str.strip().str.upper()
df["split"] = df["split"].astype(str).str.strip().str.lower()

train_mask = (df["dataset"] == "TOEFL11") & (df["split"] == "train")
train_texts = df.loc[train_mask, "masked_text"]

if train_texts.empty:
    raise ValueError("No TOEFL11 train rows found. Cannot fit vectorizers.")

print(f"Fitting vectorizers on {len(train_texts)} TOEFL11 training essays...")

word_vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 3),
    max_features=5000,
    min_df=2,
    sublinear_tf=True,
    norm="l2",
)
char_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5),
    max_features=10000,
    min_df=2,
    sublinear_tf=True,
    norm="l2",
)

word_vectorizer.fit(train_texts)
char_vectorizer.fit(train_texts)

print("Transforming all documents...")
X_word = word_vectorizer.transform(df["masked_text"])
X_char = char_vectorizer.transform(df["masked_text"])
X_combined = sparse.hstack([X_word, X_char], format="csr")

feature_data = {
    "X": X_combined,
    "y": df["label"].to_numpy(),
    "doc_id": df["doc_id"].to_numpy(),
    "dataset_tags": df["dataset"].to_numpy(),
    "split_tags": df["split"].to_numpy(),
    "feature_blocks": {
        "word": X_word.shape[1],
        "char": X_char.shape[1],
    },
}

with (FEATURES_DIR / "word_vectorizer.pkl").open("wb") as f:
    pickle.dump(word_vectorizer, f)
with (FEATURES_DIR / "char_vectorizer.pkl").open("wb") as f:
    pickle.dump(char_vectorizer, f)
with MATRICES_PATH.open("wb") as f:
    pickle.dump(feature_data, f)

print(f"Saved word vectorizer: {FEATURES_DIR / 'word_vectorizer.pkl'}")
print(f"Saved char vectorizer: {FEATURES_DIR / 'char_vectorizer.pkl'}")
print(f"Saved feature matrix: {MATRICES_PATH}")
print(f"Feature matrix shape: {X_combined.shape}")
print(f"Word features: {X_word.shape[1]}; char features: {X_char.shape[1]}")
